# 14 · Audio Degradation Pipeline Verification
Verify the acoustic degradation pipeline (low-pass filtering at 4 kHz, additive white noise at 10–20 dB SNR, and mild clipping to 90% peak) for robustness evaluation.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal as signal

from src.preprocessing.audio import SR, load_fixed
from src.preprocessing.degrade import degrade, degrade_file, degrade_intermediates

print('Successfully imported degrade, degrade_file, and degrade_intermediates from src.preprocessing.degrade')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported degrade, degrade_file, and degrade_intermediates from src.preprocessing.degrade


In [2]:
# Cell 2 – verify sha256 of labels.csv, load, and sample 20 clips from fold 10
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
labels_sha = compute_sha256(labels_path)
print(f'labels.csv sha256: {labels_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'

labels = pd.read_csv(labels_path)
f10 = labels[(labels['source'] == 'us8k') & (labels['fold'] == 10)]

# Pick 20 clips from fold 10 (us8k), 4 per app_class, random_state=42
sample_rows = []
for cls, grp in f10.groupby('app_class'):
    sample_rows.append(grp.sample(4, random_state=42))
sample_20 = pd.concat(sample_rows).reset_index(drop=True)

print(f'Sampled {len(sample_20)} clips from fold 10 (4 per app_class):')
print(sample_20['app_class'].value_counts().to_string())

assert len(sample_20) == 20, f'Expected 20 clips, got {len(sample_20)}'
assert (sample_20['app_class'].value_counts() == 4).all()
assert (sample_20['fold'] == 10).all()
assert (sample_20['source'] == 'us8k').all()
print('Cell 2 asserts passed ✓')

labels.csv sha256: 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
Sampled 20 clips from fold 10 (4 per app_class):
app_class
construction         4
hvac                 4
music_loudspeaker    4
siren                4
traffic_horn         4
Cell 2 asserts passed ✓


In [3]:
# Cell 3 – degrade the 20 clips and assert shapes, dtype, and validity
y_orig_list = []
y_deg_list = []

for i, row in sample_20.iterrows():
    path = REPO_ROOT / row['path']
    y = load_fixed(path)
    rng = np.random.default_rng(i)
    yd = degrade(y, SR, rng)
    
    assert yd.dtype == np.float32, f'Clip {i}: expected float32, got {yd.dtype}'
    assert len(yd) == 88200, f'Clip {i}: expected length 88200, got {len(yd)}'
    assert not np.isnan(yd).any(), f'Clip {i}: NaN detected'
    assert not np.isinf(yd).any(), f'Clip {i}: Inf detected'
    
    y_orig_list.append(y)
    y_deg_list.append(yd)

print(f'Successfully processed and verified all {len(sample_20)} clips (dtype float32, length 88200, no NaN/inf)')

# Print summary table of sampled clips
summary_records = []
for i, row in sample_20.iterrows():
    summary_records.append({
        'clip': i,
        'app_class': row['app_class'],
        'orig_min': float(np.min(y_orig_list[i])),
        'orig_max': float(np.max(y_orig_list[i])),
        'deg_min': float(np.min(y_deg_list[i])),
        'deg_max': float(np.max(y_deg_list[i]))
    })
df_summary = pd.DataFrame(summary_records).set_index('clip')
print('\n=== Degraded Clips Summary ===')
print(df_summary.round(4).to_string())
print('Cell 3 asserts passed ✓')

Successfully processed and verified all 20 clips (dtype float32, length 88200, no NaN/inf)

=== Degraded Clips Summary ===
              app_class  orig_min  orig_max  deg_min  deg_max
clip                                                         
0          construction   -0.0963    0.0850  -0.0623   0.0558
1          construction   -0.2368    0.1835  -0.1984   0.1819
2          construction   -0.7867    0.6395  -0.7038   0.5806
3          construction   -0.4708    0.4807  -0.4316   0.4347
4                  hvac   -0.3456    0.2924  -0.3083   0.2966
5                  hvac   -0.2572    0.2851  -0.2576   0.2576
6                  hvac   -0.3847    0.3668  -0.3592   0.3592
7                  hvac   -0.3036    0.2848  -0.2832   0.2832
8     music_loudspeaker   -0.6933    0.6869  -0.6569   0.6569
9     music_loudspeaker   -0.0610    0.0541  -0.0535   0.0527
10    music_loudspeaker   -0.5292    0.5012  -0.4495   0.4495
11    music_loudspeaker   -0.4378    0.3824  -0.4135   0.3806
12       

In [4]:
# Cell 4 – mathematical checks: low-pass drop >= 20 dB, SNR in [10, 20] dB, clipping <= 0.9 * peak
# 1. Low-pass check: compare average power spectrum above 6 kHz
freqs = np.fft.rfftfreq(88200, d=1.0 / SR)
mask_above_6k = (freqs >= 6000)

lp_drops = []
for i in range(20):
    y = y_orig_list[i]
    y_filt, _, _ = degrade_intermediates(y, SR, np.random.default_rng(i))
    
    P_orig = np.abs(np.fft.rfft(y)) ** 2
    P_filt = np.abs(np.fft.rfft(y_filt)) ** 2
    
    mean_p_orig = float(np.mean(P_orig[mask_above_6k]))
    mean_p_filt = float(np.mean(P_filt[mask_above_6k]))
    
    drop_db = 10.0 * np.log10((mean_p_orig + 1e-12) / (mean_p_filt + 1e-12))
    lp_drops.append(drop_db)

avg_lp_drop = float(np.mean(lp_drops))
print(f'Average power spectrum drop above 6 kHz: {avg_lp_drop:.2f} dB')
assert avg_lp_drop >= 20.0, f'Low-pass drop {avg_lp_drop:.2f} dB is less than 20 dB'

# 2. SNR check: for 5 clips, measure SNR between filtered signal and (noisy minus filtered)
print('\n=== SNR Verification (5 clips) ===')
for i in range(5):
    y = y_orig_list[i]
    rng = np.random.default_rng(100 + i)
    y_filt, y_noisy, _ = degrade_intermediates(y, SR, rng)
    
    noise = y_noisy - y_filt
    p_sig = float(np.mean(y_filt ** 2))
    p_noise = float(np.mean(noise ** 2))
    
    measured_snr = 10.0 * np.log10(p_sig / p_noise)
    print(f'Clip {i} | Measured SNR: {measured_snr:.2f} dB')
    assert 9.5 <= measured_snr <= 20.5, f'Clip {i} SNR {measured_snr:.2f} dB outside [10, 20] dB (+/- 0.5)'

# 3. Clipping check: assert max abs of output <= 0.9 * max abs of pre-clip noisy signal + 1e-6
print('\n=== Clipping Verification (20 clips) ===')
clipped_fractions = []
for i in range(20):
    y = y_orig_list[i]
    rng = np.random.default_rng(i)
    _, y_noisy, y_clipped = degrade_intermediates(y, SR, rng)
    
    peak_noisy = float(np.max(np.abs(y_noisy)))
    max_output = float(np.max(np.abs(y_clipped)))
    
    assert max_output <= 0.9 * peak_noisy + 1e-6, f'Clip {i}: max abs {max_output:.6f} > 0.9 * {peak_noisy:.6f} + 1e-6'
    
    frac_clipped = float(np.mean(np.abs(y_noisy) > 0.9 * peak_noisy))
    clipped_fractions.append(frac_clipped)

avg_clipped_frac = float(np.mean(clipped_fractions))
print(f'Average fraction of clipped samples across 20 clips: {avg_clipped_frac:.4%}')
print(f'Per-clip clipped fraction range: [{min(clipped_fractions):.4%}, {max(clipped_fractions):.4%}]')
print('Cell 4 asserts passed ✓')

Average power spectrum drop above 6 kHz: 27.10 dB

=== SNR Verification (5 clips) ===
Clip 0 | Measured SNR: 18.35 dB
Clip 1 | Measured SNR: 19.41 dB
Clip 2 | Measured SNR: 11.66 dB
Clip 3 | Measured SNR: 13.15 dB
Clip 4 | Measured SNR: 18.38 dB

=== Clipping Verification (20 clips) ===
Average fraction of clipped samples across 20 clips: 0.0240%
Per-clip clipped fraction range: [0.0011%, 0.1905%]
Cell 4 asserts passed ✓


In [5]:
# Cell 5 – verify determinism with degrade_file
test_paths = [REPO_ROOT / sample_20.iloc[k]['path'] for k in [0, 1, 2]]

for k, p in enumerate(test_paths):
    deg_a = degrade_file(p, seed=7)
    deg_b = degrade_file(p, seed=7)
    deg_c = degrade_file(p, seed=8)
    
    assert np.array_equal(deg_a, deg_b), f'Path {k}: degrade_file with same seed produced different results'
    assert not np.array_equal(deg_a, deg_c), f'Path {k}: degrade_file with different seeds produced identical results'
    
    diff_seed = float(np.max(np.abs(deg_a - deg_c)))
    print(f'Path {k}: seed 7 == seed 7 exact match; seed 7 vs seed 8 max diff = {diff_seed:.6f}')

print('\nCell 5 asserts passed: deterministic seeding verified ✓')

Path 0: seed 7 == seed 7 exact match; seed 7 vs seed 8 max diff = 0.008048
Path 1: seed 7 == seed 7 exact match; seed 7 vs seed 8 max diff = 0.066728
Path 2: seed 7 == seed 7 exact match; seed 7 vs seed 8 max diff = 0.099302

Cell 5 asserts passed: deterministic seeding verified ✓


In [6]:
# Cell 6 – save 3 before/after spectrogram-style plots (waveform and log-magnitude spectrum)
fig_dir = REPO_ROOT / 'reports/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / 'degrade_examples.png'

fig, axes = plt.subplots(3, 2, figsize=(14, 9))

time_axis = np.linspace(0, 4.0, 88200)
freq_axis = np.fft.rfftfreq(88200, d=1.0 / SR)

for row_idx in range(3):
    y_orig = y_orig_list[row_idx]
    y_deg = y_deg_list[row_idx]
    cls_name = sample_20.iloc[row_idx]['app_class']
    
    # Left column: Waveform comparison
    ax_wave = axes[row_idx, 0]
    ax_wave.plot(time_axis, y_orig, label='Original', color='steelblue', alpha=0.7)
    ax_wave.plot(time_axis, y_deg, label='Degraded', color='crimson', alpha=0.6, lw=0.8)
    ax_wave.set_title(f'Clip {row_idx}: {cls_name} – Waveform')
    ax_wave.set_xlabel('Time (s)')
    ax_wave.set_ylabel('Amplitude')
    ax_wave.grid(True, alpha=0.3)
    ax_wave.legend(loc='upper right')
    
    # Right column: Log-magnitude spectrum
    ax_spec = axes[row_idx, 1]
    P_orig = np.abs(np.fft.rfft(y_orig))
    P_deg = np.abs(np.fft.rfft(y_deg))
    
    mag_orig_db = 20 * np.log10(P_orig + 1e-9)
    mag_deg_db = 20 * np.log10(P_deg + 1e-9)
    
    ax_spec.plot(freq_axis, mag_orig_db, label='Original', color='steelblue', alpha=0.7)
    ax_spec.plot(freq_axis, mag_deg_db, label='Degraded', color='crimson', alpha=0.6)
    ax_spec.axvline(4000, color='black', linestyle='--', alpha=0.6, label='LP Cutoff (4 kHz)')
    ax_spec.axvline(6000, color='gray', linestyle=':', alpha=0.6, label='6 kHz')
    ax_spec.set_title(f'Clip {row_idx}: {cls_name} – Log-Magnitude Spectrum')
    ax_spec.set_xlabel('Frequency (Hz)')
    ax_spec.set_ylabel('Magnitude (dB)')
    ax_spec.set_xlim(0, SR / 2)
    ax_spec.grid(True, alpha=0.3)
    ax_spec.legend(loc='upper right')

plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.close(fig)

print(f'Saved degrade examples plot to: {fig_path}')
print('Cell 6 completed ✓')

Saved degrade examples plot to: C:\Users\LENOVO\Desktop\CityPulse\reports\figures\degrade_examples.png
Cell 6 completed ✓
